## 1 setup

In [3]:
%pip install -q "datasets==3.6.0" pandas pyarrow numpy tqdm

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
from pathlib import Path
import json
import math
import warnings #gestire warnings
import numpy as np 
import pandas as pd
import pyarrow.parquet as pq 
from tqdm.auto import tqdm #barra di progresso

warnings.filterwarnings("ignore", category=FutureWarning) #nasconde avvisi per avere output meno rumoroso
#impostazioni di pandas
pd.set_option("display.max_columns",100)
pd.set_option("display.max_colwidth",140)
pd.set_option("display.float_format", lambda x: f"{x:,.6f}")  

## 2 percorsi e configurazione scelta nel notebook 02

In [5]:
CATEGORY="Electronics" #categoria scelta
SELECTED_CONFIG="user3_item8" #configurazione scelta
POSITIVE_RATING_THRESHOLDS=4 #rating minimo per considerare una interazione positiva
MIN_USER_INTERACTIONS=3 
MIN_ITEM_INTERACTIONS=8
N_NEG=100 #quanti item negativi campionare per ogni esempio di validation/test
RANDOM_STATE=42 #seed casuale per rendere il negative sampling riproducibile
BUILD_NEGATIVES_SAMPLES=True  #se diventa troppo lento, puoi metterlo a false e saltare il negative sampling
RUN_POPULARITY_BASELINE=True #se è True, valuta una baseline Most Popular per controllare la pipeline


In [6]:
PROJECT_ROOT=Path("..").resolve() if (Path("..")/"data").exists() else Path(".").resolve() #ROOT progetto
SOURCE_DIR=PROJECT_ROOT/"data"/"processed"/"electronics_intersection" #output dei precedenti notebook
ANALYSIS_DIR=SOURCE_DIR/"analysis" #riepiloghi notebook 2
OUTPUT_DIR=PROJECT_ROOT/"data"/"processed"/"electronics_recommender" #output di questo notebook03
OUTPUT_DIR.mkdir(parents=True,exist_ok=True) #crea cartella output se non esiste

CANDIDATE_INTERACTIONS_PATH=SOURCE_DIR/f"candidate_interactions_{SELECTED_CONFIG}.parquet" #interazioni candidate
CANDIDATE_ITEMS_PATH=SOURCE_DIR/f"candidate_items_{SELECTED_CONFIG}.parquet" #prodotti candidati
CANDIDATE_QA_PATH=SOURCE_DIR/f"candidate_qa_{SELECTED_CONFIG}.parquet"  #qa candidati
NOTEBOOK02_CONFIG_PATH=ANALYSIS_DIR/"notebook02_selected_filter_config.json" #riepilogo config scelta

input_paths={ #dizionario coi file di input
    "candidate_interactions":CANDIDATE_INTERACTIONS_PATH,
    "candidate_items":CANDIDATE_ITEMS_PATH,
    "candidate_qa":CANDIDATE_QA_PATH,
    "notebook02_config":NOTEBOOK02_CONFIG_PATH,
}

print("Project root:",PROJECT_ROOT)
print("Source dir:",SOURCE_DIR)
print("Output dir:",OUTPUT_DIR)
for name,path in input_paths.items():
    print(f"{name:24s}->{path} | exists={path.exists()}")

Project root: C:\Users\arsid\Desktop\Tesi-ecommerce
Source dir: C:\Users\arsid\Desktop\Tesi-ecommerce\data\processed\electronics_intersection
Output dir: C:\Users\arsid\Desktop\Tesi-ecommerce\data\processed\electronics_recommender
candidate_interactions  ->C:\Users\arsid\Desktop\Tesi-ecommerce\data\processed\electronics_intersection\candidate_interactions_user3_item8.parquet | exists=True
candidate_items         ->C:\Users\arsid\Desktop\Tesi-ecommerce\data\processed\electronics_intersection\candidate_items_user3_item8.parquet | exists=True
candidate_qa            ->C:\Users\arsid\Desktop\Tesi-ecommerce\data\processed\electronics_intersection\candidate_qa_user3_item8.parquet | exists=True
notebook02_config       ->C:\Users\arsid\Desktop\Tesi-ecommerce\data\processed\electronics_intersection\analysis\notebook02_selected_filter_config.json | exists=True


In [7]:
missing_paths=[str(path) for path in input_paths.values() if not path.exists()] #lista file mancanti
if missing_paths:
    raise FileNotFoundError(
        "Mancano alcuni file prodotti dal Notebook 02:\n"
        +"\n".join(missing_paths)
        +"\n\nEsegui prima il Notebook 02 fino al salvataggio del dataset candidato."
    )

## 3 caricamento dei dati candidati

In [10]:
#carichiamo dati
interactions_df=pd.read_parquet(CANDIDATE_INTERACTIONS_PATH)
items_df=pd.read_parquet(CANDIDATE_ITEMS_PATH)
qa_df=pd.read_parquet(CANDIDATE_QA_PATH)
notebook02_Config=json.loads(NOTEBOOK02_CONFIG_PATH.read_text(encoding="utf-8"))
#stampiamo dimensioni tabelle
print("interactions_df:",interactions_df.shape)
print("items_df:",items_df.shape)
print("qa_df:",qa_df.shape)
display(pd.DataFrame([notebook02_Config])) #riepilogo dati dentro configurazione scelta

interactions_df: (844060, 7)
items_df: (15069, 12)
qa_df: (128151, 8)


,selected_config_name,min_user_interactions,min_item_interactions,num_users,num_items,num_interactions,density_pct,sparsity_pct
0,user3_item8,3,8,196509,15069,844060,0.028504,99.971496


## 4 prepariamo timestamp

In [16]:
TIME_COL="datetime" #colonna temporale usata nello split

if TIME_COL in interactions_df.columns: 
    interactions_df[TIME_COL]=pd.to_datetime( #convertiamola in datetime pandas 
        interactions_df[TIME_COL], 
        errors="coerce", #trasforma errori in NaT
    )
elif "timestamp" in interactions_df.columns: #se datetime non c'è, usa timestamp
    interactions_df[TIME_COL]=pd.to_datetime(
        interactions_df["timestamp"],
        unit="ms",
        errors="coerce",
    )
else:
    raise ValueError("mancano sia timestamp sia datetime")

invalid_datetime_rows=int(interactions_df[TIME_COL].isna().sum()) #conta righe senza datetime valido
if invalid_datetime_rows>0:
    print("righe rimosse senza datetime valido:", invalid_datetime_rows)
    interactions_df=interactions_df.dropna(subset=[TIME_COL]).copy() #rimuove righe


print("Colonna temporale usata per lo split:",TIME_COL)
print("Data minima:",interactions_df[TIME_COL].min())
print("Data massima:",interactions_df[TIME_COL].max()) 

preview_c=["timestamp",TIME_COL] if "timestamp" in interactions_df.columns else [TIME_COL]
display(interactions_df[preview_c].head())

Colonna temporale usata per lo split: datetime
Data minima: 1999-08-21 22:04:38
Data massima: 2023-06-30 05:28:46.120000


,timestamp,datetime
0,1275251437000,2010-05-30 20:30:37
1,1311718723000,2011-07-26 22:18:43
2,1344043262000,2012-08-04 01:21:02
3,1389108747000,2014-01-07 15:32:27
4,1381408252000,2013-10-10 12:30:52


## 5 definizione target: feedback positivo
consideriamo coppie come rilevanti se rating è almeno 4

In [20]:
rating_dist=( 
    #distribuzione dei rating nel dataset candidato, la rifacciamo perchè potrebbe essere
    #diverso dopo aver filtrato con la configurazione scelta
    interactions_df["rating"]
    .value_counts(dropna=False)
    .sort_index()
    .rename_axis("rating")
    .reset_index(name="count")
)
rating_dist["percentage"]=rating_dist["count"]/len(interactions_df)*100
display(rating_dist)

,rating,count,percentage
0,1.000000,61551,7.292254
1,2.000000,41741,4.945265
2,3.000000,68865,8.158780
3,4.000000,152080,18.017676
4,5.000000,519823,61.586025


In [ ]:
positive_interactions_df=interactions_df[interactions_df["rating"]]>= POSITIVE_RATING_THRESHOLD.copy() #tiene solo rating pos
positive_interactions_df["interaction_value"]=1 #crea colonna binaria cosi capiamo i rating positivi
target_stats={
    "original_interaction":int(len(interactions_df)), #numero di interazioni candidate originali
    "positive_interaction":int(len(positive_interactions_df)), #n int con rating >=soglia
    "removed_interaction":int(len(interactions_df)-len(positive_interactions_df)), #num int escluse
    "positive_int_percentage":float(len(positive_interactions_df)/len(interaction_df)*100),
    "original_users_num":int(interaction_df["user_id"].nunique()),
    "positive_users_num":int(positive_interactions_df["user_id"].nunique()),
    "original_items":int(interactions_df["parent_asin"].nunique()),
    "positive_items":int(positive_interactions_df["parent_asin"].nunique),
}

target_stats_df=pd.DataFrame.from_dict(target_stats,orient="index",columns=["value"])
display(target_stats_df)